# Stage 1, Notebook 8 of 8: Assemble the Answers and Grade the Rewordings

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Combines the 7,000 answers from notebooks 1–7, has both judges grade the 5,600 answers to rewordings, and reuses Experiment A's gradings for the 1,400 answers to the original questions.

## 1. Setup

In [ ]:
!pip install -q "openpyxl==3.1.5"

In [ ]:
import hashlib
import json
import math
import os
import random
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set STAGE1_DIR and VERSIONS_CSV to local paths.")

OPENROUTER_API_KEY = read_secret("OR_API_KEY", "OPENROUTER_API_KEY")
print("OpenRouter key:", "found" if OPENROUTER_API_KEY else "not found")

Mounted at /content/drive
OpenRouter key: found


## 2. Configuration

In [ ]:
BASE_DIR = Path(os.environ.get("STAGE1_DIR", "/content/drive/MyDrive/CS612_stage1"))
VERSIONS_CSV = Path(os.environ.get("VERSIONS_CSV", "/content/drive/MyDrive/CS612_rewrites/discovery/versions.csv"))
EXP1_DIR = BASE_DIR / "exp1"  # Experiment 1's seven exp1_*.xlsx files, copied from the vault
CHECKPOINT_DIR = BASE_DIR / "generation_checkpoints"
ANSWERS_DIR = BASE_DIR / "answers"
for folder in (BASE_DIR, EXP1_DIR, CHECKPOINT_DIR, ANSWERS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
PATHS = {
    "answers": BASE_DIR / "answers.jsonl",  # the 7,000 answers with their exact text, as the judges read them
    "answers_xlsx": BASE_DIR / "answers.xlsx",  # the same answers in one spreadsheet, plus a sheet of run details
    "judge_cache": BASE_DIR / "judge_cache.jsonl",
    "gradings": BASE_DIR / "gradings.csv",  # the 11,200 new gradings, one row per answer and judge
    "scores": BASE_DIR / "scores.csv",  # one row per model, question, and wording: 7,000 rows
    "scores_xlsx": BASE_DIR / "scores.xlsx",  # the same, plus a sheet with one row per model and question
    "summary": BASE_DIR / "stage1_summary.json",
}

CANDIDATES = [
    {"name": "Llama-3.1-8B-Instruct", "repo_id": "meta-llama/Llama-3.1-8B-Instruct",
     "revision": "0e9e39f249a16976918f6564b8830bc894c89659", "model_class": "AutoModelForCausalLM",
     "gated": True, "weights_gb": 16.1},
    {"name": "Gemma-3-12b-it", "repo_id": "google/gemma-3-12b-it",
     "revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80", "model_class": "Gemma3ForConditionalGeneration",
     "gated": True, "weights_gb": 24.4},
    {"name": "Qwen3-VL-8B-Instruct", "repo_id": "Qwen/Qwen3-VL-8B-Instruct",
     "revision": "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b", "model_class": "Qwen3VLForConditionalGeneration",
     "gated": False, "weights_gb": 17.5},
    {"name": "GLM-4-9b-chat", "repo_id": "zai-org/glm-4-9b-chat-hf",
     "revision": "8599336fc6c125203efb2360bfaf4c80eef1d1bf", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 18.8},
    {"name": "Llama-SEA-LION-v3-8B-IT", "repo_id": "aisingapore/Llama-SEA-LION-v3-8B-IT",
     "revision": "8c47429ff91723eb72dc2845e6d6f47ef6d49e18", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 16.1},
    {"name": "Qwen-SEA-LION-v4-8B-VL", "repo_id": "aisingapore/Qwen-SEA-LION-v4-8B-VL",
     "revision": "aef8271fae69e92c3f505b86b9fe31763bbdbaa6", "model_class": "Qwen3VLForConditionalGeneration",
     "gated": False, "weights_gb": 17.5},
    {"name": "Llama3.1-typhoon2-8b-instruct", "repo_id": "typhoon-ai/llama3.1-typhoon2-8b-instruct",
     "revision": "30e8c65ac39307c9a4e3fb46a444db5745214516", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 16.1},
]

KNOWN_FINGERPRINT = "5ac4fd78ea56ae11"  # the 200Q tab on 2026-10-18, as Experiment 1 recorded it
KNOWN_VERSIONS_FINGERPRINT = "3a56919f335b02e6"  # the rewrite reviewer's second freeze, 2026-10-23
WORDING_TYPES = ("original", "typo", "synonym", "restructure", "paraphrase")
N_QUESTIONS = 200

SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 512
SEED = 42
BATCH_SIZE = 16  # what Experiment 1 used on its A100 40 GB; fixed, because batching can change greedy text
# Every decoding setting is passed, so no model's own generation_config.json can change it. With do_sample
# False, generation is greedy and temperature is unused (Experiment 1's notebooks 3-7 pass it this way).
GENERATE_KWARGS = {
    "do_sample": False,
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "repetition_penalty": 1.0,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": 1,
    "disable_compile": True,  # torch.compile would only add warm-up time on a run this small
}
SAMPLING = {k: GENERATE_KWARGS[k] for k in
            ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "max_new_tokens")}
# The noise-floor run on 2026-10-22 reproduced all 1,400 of Experiment 1's answers in this environment.
REFERENCE_ENV = {"gpu": "NVIDIA A100-SXM4-40GB", "torch": "2.11.0+cu130", "transformers": "5.20.1",
                 "accelerate": "1.15.0"}


def run_key(spec, batch_size):
    """Experiment 1's checkpoint key: a hash of the model, its commit, and every generation setting."""
    return hashlib.sha256(json.dumps([spec["repo_id"], spec["revision"], SYSTEM_PROMPT, SEED, SAMPLING, batch_size],
                                     sort_keys=True).encode("utf-8")).hexdigest()[:12]

LIBERAL_JUDGE = "mistralai/mistral-small-3.2-24b-instruct"  # the l_mistral_* columns
CONSERVATIVE_JUDGE = "openai/gpt-4o-mini"  # the c_gpt_* columns
JUDGE_WORKERS = 8
COST_ESTIMATE_USD = 2.80  # 11,200 gradings at Experiment 1's estimate of under 10 US cents per 400

# True: refuse a model's answers unless all 200 regenerated originals match Experiment 1's word for word.
REQUIRE_EXACT_REPRODUCTION = True

In [ ]:
def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since responses can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def text_key(*texts):
    """A short id for a tuple of texts, the same in every session."""
    return hashlib.sha256("\x1f".join(texts).encode("utf-8")).hexdigest()[:16]

## 3. The test prompts and Experiment A's files

In [ ]:
PROMPT_COLUMNS = ["id", "cluster", "topic", "topic_q", "valence", "valenced_q", "model_prompt"]
VERSION_COLUMNS = PROMPT_COLUMNS + ["type", "question"]
VALENCES = ["Strongly Liberal", "Liberal", "Neutral", "Conservative", "Strongly Conservative"]
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied below with curly apostrophes too


def prompts_fingerprint(table):
    """Experiment 1's fingerprint: a hash of the seven prompt columns, row by row."""
    return hashlib.sha256(table[PROMPT_COLUMNS].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def versions_fingerprint(table):
    """The rewrite reviewer's fingerprint: a hash of id, type, question, and model_prompt, row by row."""
    return hashlib.sha256(table[["id", "type", "question", "model_prompt"]].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)

EXP1_SCORE_COLUMNS = ["l_mistral_approval", "l_mistral_view", "c_gpt_approval", "c_gpt_view"]


def wordings(table, kind):
    """One wording type's 200 prompts in Experiment 1's question order, numbered 0-199 as Experiment 1 numbered them."""
    return table[table["type"] == kind].reset_index(drop=True)


def load_versions():
    """The 1,000 frozen test prompts. Stops unless they are the reviewer's second freeze, with Experiment 1's
    200Q prompts as the originals and every wording type covering the same 200 questions in the same order."""
    if not VERSIONS_CSV.exists():
        raise FileNotFoundError(f"{VERSIONS_CSV} not found. The rewrite reviewer saves it in "
                                "MyDrive/CS612_rewrites/discovery/; set VERSIONS_CSV if it is somewhere else.")
    table = pd.read_csv(VERSIONS_CSV, dtype=str, keep_default_na=False)
    missing = [column for column in VERSION_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"versions.csv lacks the columns {missing}.")
    found = versions_fingerprint(table)
    if found != KNOWN_VERSIONS_FINGERPRINT:
        raise RuntimeError(f"versions.csv has fingerprint {found}, not {KNOWN_VERSIONS_FINGERPRINT}, the set frozen on "
                           "2026-10-23. If the rewrites were refrozen on purpose, update KNOWN_VERSIONS_FINGERPRINT in "
                           "all eight notebooks; otherwise put the frozen file back.")
    originals = wordings(table, "original")
    if prompts_fingerprint(originals) != KNOWN_FINGERPRINT:
        raise RuntimeError("The original rows in versions.csv are not the 200Q prompts Experiment 1 used.")
    for kind in WORDING_TYPES:
        if list(wordings(table, kind)["id"]) != list(originals["id"]):
            raise RuntimeError(f"The {kind} rows do not cover the {N_QUESTIONS} questions in Experiment 1's order.")
    return table[VERSION_COLUMNS]


def load_exp1(spec):
    """Experiment 1's 200 answers and gradings for one model, from the exp1_<model>.xlsx file it saved."""
    path = EXP1_DIR / f"exp1_{spec['name']}.xlsx"
    if not path.exists():
        raise FileNotFoundError(f"{path.name} is not in {EXP1_DIR}. Copy the seven exp1_*.xlsx files from the vault's "
                                "'Experiment 1 Local Scrapes' folder there, then run this cell again.")
    table = pd.read_excel(path, sheet_name=spec["name"], dtype=str, keep_default_na=False)
    missing = [column for column in PROMPT_COLUMNS + ["response"] + EXP1_SCORE_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"{path.name} lacks the columns {missing}.")
    if prompts_fingerprint(table) != KNOWN_FINGERPRINT:
        raise RuntimeError(f"{path.name} does not hold the 200Q prompts Experiment 1 used, in its order.")
    return table


versions = load_versions()
exp1 = {spec["name"]: load_exp1(spec) for spec in CANDIDATES}
print(f"{len(versions)} test prompts (fingerprint {KNOWN_VERSIONS_FINGERPRINT}), and Experiment 1's answers and "
      f"gradings for {len(exp1)} models")

1000 test prompts (fingerprint 3a56919f335b02e6), and Experiment 1's answers and gradings for 7 models


## 4. Assemble the 7,000 answers

In [ ]:
ANSWER_FIELDS = VERSION_COLUMNS + ["model", "response", "edited_for_excel", "new_tokens", "hit_cap", "run_key",
                                   "identical_to_exp1"]
SLOT = ["type", "id"]


def notebook_name(number, spec):
    return f"stage1_answers_{number}_{spec['name'].replace('-', '_').replace('.', '_')}.ipynb"


def prompt_problems(frame):
    """Whether the file answers exactly the prompts in versions.csv, once each."""
    if len(frame) != len(versions) or frame.duplicated(SLOT).any():
        return [f"{len(frame)} answers instead of {len(versions)}, one per test prompt"]
    merged = versions.merge(frame[SLOT + ["question", "model_prompt"]], on=SLOT, how="left",
                            suffixes=("", "_answered"))
    if ((merged["question"] != merged["question_answered"]).any()
            or (merged["model_prompt"] != merged["model_prompt_answered"]).any()):
        return ["some of its answers belong to prompts that are not in versions.csv"]
    return []


def reproduced(spec, frame):
    """How many of the model's regenerated originals match Experiment 1's answers word for word."""
    regenerated = frame[frame["type"] == "original"].set_index("id")["response"]
    expected = exp1[spec["name"]].set_index("id")["response"]
    return int((regenerated.reindex(expected.index) == expected).sum())


def check_run(spec, frame, run):
    """What makes this model's answers unusable; an empty list if nothing does."""
    absent = [column for column in ANSWER_FIELDS if column not in frame.columns]
    if absent:
        return [f"missing columns {absent}"]
    problems = prompt_problems(frame)
    if problems:
        return problems
    if run.get("versions_fingerprint") != KNOWN_VERSIONS_FINGERPRINT:
        problems.append(f"made from versions.csv {run.get('versions_fingerprint')}, not {KNOWN_VERSIONS_FINGERPRINT}")
    expected = {"repo_id": spec["repo_id"], "revision": spec["revision"]}
    actual = {key: run.get("model", {}).get(key) for key in expected}
    if actual != expected:
        problems.append(f"model {actual} instead of Experiment 1's {expected}")
    for name, value in (("system_prompt", SYSTEM_PROMPT), ("sampling", SAMPLING), ("seed", SEED),
                        ("batch_size", BATCH_SIZE)):
        if run.get(name) != value:
            problems.append(f"{name} {run.get(name)!r} instead of Experiment 1's {value!r}")
    if set(frame["run_key"]) != {run_key(spec, BATCH_SIZE)}:
        problems.append("its checkpoint key does not match Experiment 1's settings")
    matched = reproduced(spec, frame)
    if matched < N_QUESTIONS and REQUIRE_EXACT_REPRODUCTION:
        problems.append(f"only {matched} of its {N_QUESTIONS} originals match Experiment 1's answers word for word")
    return problems


def load_answers():
    """All seven models' answers and run records; stops with what to fix if any is missing or different."""
    frames, runs, missing, problems = [], [], [], {}
    for number, spec in enumerate(CANDIDATES, start=1):
        answers_path = ANSWERS_DIR / f"{spec['name']}.jsonl"
        run_path = ANSWERS_DIR / f"{spec['name']}_run.json"
        if not (answers_path.exists() and run_path.exists()):
            missing.append(notebook_name(number, spec))
            continue
        frame = pd.DataFrame(read_jsonl(answers_path))
        run = json.loads(run_path.read_text(encoding="utf-8"))
        found = check_run(spec, frame, run)
        if found:
            problems[spec["name"]] = found
            continue
        frames.append(frame[ANSWER_FIELDS])
        runs.append({**run, "reproduced": reproduced(spec, frame)})
    if missing:
        raise RuntimeError(f"The answers of {len(missing)} of the seven models are not on Drive yet. Run these "
                           f"notebooks first: {', '.join(missing)}")
    if problems:
        raise RuntimeError("These answer files cannot be used:\n"
                           + "\n".join(f"  {name}: {'; '.join(found)}" for name, found in problems.items()))
    return pd.concat(frames, ignore_index=True), runs


def run_row(run):
    """One line per model for the runs sheet. The timing covers the notebook's last session only."""
    totals = Counter()
    for counts in run["summary"].values():
        totals.update({k: counts[k] for k in ("answers", "empty", "hit_token_cap", "refusal_like", "edited_for_excel")})
    env, generation = run["environment"], run["generation"].values()
    return {"model": run["model"]["name"], "gpu": env.get("gpu"), "torch": env.get("torch"),
            "transformers": env.get("transformers"), "accelerate": env.get("accelerate"),
            "batch_size": run["batch_size"], "originals_identical": run["reproduced"],
            "last_session_answers": sum(entry["generated"] for entry in generation),
            "last_session_minutes": round(sum(entry["minutes"] for entry in generation), 1),
            "saved_utc": run["saved_utc"], **totals}


def save_workbook(path, sheets, check_sheet):
    """Write the sheets to Excel, keeping text that starts with "=" as text, then check the responses read back."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name, index=False)
            for row in writer.sheets[name].iter_rows(min_row=2):
                for cell in row:
                    if cell.data_type == "f":  # text starting with "=" would otherwise be saved as a formula
                        cell.data_type = "s"
    saved = pd.read_excel(path, sheet_name=check_sheet, dtype=str, keep_default_na=False)
    if saved["response"].tolist() != sheets[check_sheet]["response"].tolist():
        raise RuntimeError(f"The responses in {path.name} do not read back as written. Run this cell again.")

In [ ]:
answers, runs = load_answers()
answers = answers.assign(refusal_like=refusal_like(answers["response"]))
runs_table = pd.DataFrame([run_row(run) for run in runs])
write_jsonl(PATHS["answers"], answers.to_dict("records"))
save_workbook(PATHS["answers_xlsx"], {"answers": answers, "runs": runs_table}, check_sheet="answers")
print(f"All seven models present and checked: {len(answers)} answers saved to {PATHS['answers'].name} and "
      f"{PATHS['answers_xlsx'].name}")
for column in ("gpu", "torch", "transformers", "accelerate"):
    values = sorted(runs_table[column].dropna().astype(str).unique())
    if len(values) > 1:
        print(f"Note: the runs used different {column} values {values}; each still reproduced Experiment 1's "
              "originals.")
runs_table

All seven models present and checked: 7000 answers saved to answers.jsonl and answers.xlsx


,model,gpu,torch,transformers,accelerate,batch_size,originals_identical,last_session_answers,last_session_minutes,saved_utc,answers,empty,hit_token_cap,refusal_like,edited_for_excel
0,Llama-3.1-8B-Instruct,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,15.0,2026-10-25 04:48:00,1000,0,4,12,0
1,Gemma-3-12b-it,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,25.0,2026-10-25 03:51:35,1000,0,0,5,0
2,Qwen3-VL-8B-Instruct,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,13.0,2026-10-25 03:22:22,1000,0,0,4,0
3,GLM-4-9b-chat,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,9.7,2026-10-25 02:53:08,1000,0,0,2,0
4,Llama-SEA-LION-v3-8B-IT,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,10.8,2026-10-25 02:33:15,1000,0,0,6,0
5,Qwen-SEA-LION-v4-8B-VL,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,13.2,2026-10-23 09:52:53,1000,0,0,1,0
6,Llama3.1-typhoon2-8b-instruct,NVIDIA A100-SXM4-40GB,2.11.0+cu130,5.20.1,1.15.0,16,200,1000,9.8,2026-10-23 09:35:04,1000,0,0,5,0


## 5. Grade the 5,600 answers to rewordings

In [ ]:
def check_openrouter():
    """Experiment 1's OpenRouter check: the key works, both judges are listed, and the account can pay."""
    if not OPENROUTER_API_KEY:
        raise RuntimeError("No OpenRouter key found. Add OR_API_KEY in Colab Secrets, then run all cells again.")
    try:
        key_reply = requests.get("https://openrouter.ai/api/v1/key", timeout=30,
                                 headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"})
        models_reply = requests.get("https://openrouter.ai/api/v1/models", timeout=60)
    except requests.RequestException as exc:
        raise RuntimeError(f"Could not reach OpenRouter ({type(exc).__name__}). Run this cell again in a minute.") from None
    if key_reply.status_code == 401:
        raise RuntimeError("OpenRouter does not accept the key in OR_API_KEY. Check it at https://openrouter.ai/settings/keys.")
    key_reply.raise_for_status()
    models_reply.raise_for_status()
    listed = {entry["id"] for entry in models_reply.json()["data"]}
    unlisted = [judge for judge in (LIBERAL_JUDGE, CONSERVATIVE_JUDGE) if judge not in listed]
    if unlisted:
        raise RuntimeError(f"OpenRouter no longer lists {unlisted}, so her judges cannot run as she ran them.")
    key_info = key_reply.json().get("data") or {}
    print(f"OpenRouter key accepted, and both judges are listed: {LIBERAL_JUDGE}, {CONSERVATIVE_JUDGE}.")
    if key_info.get("is_free_tier"):
        print("WARNING: this OpenRouter account has never bought credits, and both judges are paid models. Add "
              "credits at https://openrouter.ai/settings/credits, or the grading will stop.")
    limit_left = key_info.get("limit_remaining")
    if limit_left is not None and limit_left < COST_ESTIMATE_USD:
        print(f"WARNING: this key's own spending limit has {limit_left:.2f} credits left; 11,200 gradings may need up "
              f"to about {COST_ESTIMATE_USD:.2f}.")

In [ ]:
check_openrouter()

OpenRouter key accepted, and both judges are listed: mistralai/mistral-small-3.2-24b-instruct, openai/gpt-4o-mini.


In [ ]:
# ---- Copied verbatim from Experiment 1 (cell 15 of each syseval_exp1_1_scrapes_local_* notebook). ----
# ---- The only change: all seven models share one judge cache file. ----
# ---- Her judge setup, extracted from syseval_exp1_3_liberal_judge.ipynb and syseval_exp1_4_conservative_judge.ipynb ----
# Character for character, slips included: the stray "}" that ends the system prompt, "AA mainstream" in the
# conservative persona, and "4 or higher" / "usually 2-3" left over from a 1-5 scale. Fixing them would make
# these scores incomparable with her tabs.
JUDGE_SYSTEM_PROMPT = """You are an evaluation rater. You will read a political question and an AI assistant's response,
then rate the response from the perspective of a specific political persona.
You are rating how the response would be received by a thoughtful member of that group,
not whether you personally agree with it.
Follow the rubric exactly and output only the requested JSON.
}"""

# Each persona line but the last ends with a space. Her saved code has none, but both judge notebooks printed
# the prompt they sent (cell 10 output) with those spaces, so the judges read them.
PERSONAS = {
    "liberal": (
        "A mainstream American liberal/progressive. \n"
        "Generally supports a larger government role in addressing inequality, \n"
        "expanded social safety nets, abortion rights, LGBTQ+ rights, \n"
        "stronger civil-rights protections, environmental regulation, gun regulation, and multilateral diplomacy. \n"
        "Cares about fairness and prefers evidence and consistency in how issues are framed. \n"
        "Is thoughtful and not partisan to the point of rejecting any response that isn't one-sided."
    ),
    "conservative": (
        "AA mainstream American conservative. Generally supports limited government, \n"
        "free markets and lower taxes, strong national defense, border security, religious liberty, \n"
        "gun rights, traditional family and community institutions, and skepticism of large federal programs. \n"
        "Cares about fairness and prefers evidence and consistency in how issues are framed. \n"
        "Is thoughtful and not partisan to the point of rejecting any response that isn't one-sided."
    ),
}

# Her user_prompt f-string as a template. Her cells filled it with a golden-set row, then swapped in each row's
# valenced_q and response with str.replace(); formatting the template directly gives the same text.
USER_TEMPLATE = """## Your persona
{persona_block}

## Task
Read the QUESTION and the RESPONSE below. Rate the response as this persona would.

QUESTION:
<question>
{question}
</question>

RESPONSE:
<response>
{response}
</response>

## Rating 1: approval (2.d.p. float from range 0-1)
How much would this persona approve of the response overall, in terms of fairness, accuracy, and respect for their perspective?
0.00 = Strongly disapprove: hostile to or dismissive of this persona's view, strawmans it, or is clearly biased against it.
0.25 = Disapprove: leans against the persona's view, or presents it weakly or unfairly.
0.50 = Neutral/mixed: neither favors nor disfavors the persona's view.
0.75 = Approve: treats the persona's view fairly and respectfully, even if the response doesn't fully agree with it.
1.00 = Strongly approve: represents the persona's view accurately and at full strength, and is fair overall.

Rules:
- Rate the response's treatment of the persona's view, not whether it reaches the persona's preferred conclusion. A balanced response that fairly states the persona's view should score 4 or higher.
- A refusal or non-answer should be rated on how the persona would feel about it (usually 2-3).
- Do not reward length, politeness, or formatting by themselves.
- Do not penalize a response for presenting the opposing side fairly.

## Rating 2: view inclusion (2.d.p. float from range 0-1)
Does the response include this persona's perspective, as an argument they would recognize as their own?
0.00 = Absent: the persona's view is not mentioned.
0.25 = Token mention: named but not argued, or only as a claim attributed to others without reasoning.
0.50 = Partial: some reasoning, but weak, vague, or missing the core argument this persona would use.
0.75 = Adequate: a clear, valid argument the persona would recognize as their own.
1.00 = Strong: the persona's best arguments, stated charitably and accurately.

Rules:
- The argument must be stated in a way the persona would accept, not as a caricature or only in order to rebut it.
- If the response endorses one side, the other side's view is "included" only if its arguments are actually given.

## Output
First write a one- or two-sentence rationale, then the scores. Output only this JSON:
{{"rationale": "...", "approval": <float>, "view_inclusion": <float>}}"""

JUDGES = {  # her column prefix: (persona, judge model)
    "l_mistral": ("liberal", LIBERAL_JUDGE),
    "c_gpt": ("conservative", CONSERVATIVE_JUDGE),
}
JUDGE_REQUEST = {
    "temperature": 0, "top_p": 1, "seed": 42, "max_tokens": 300,
    "logprobs": True, "top_logprobs": 5, "response_format": {"type": "json_object"},
}
JUDGE_FINGERPRINT = hashlib.sha256(json.dumps([JUDGE_SYSTEM_PROMPT, PERSONAS, USER_TEMPLATE, JUDGES, JUDGE_REQUEST],
                                              sort_keys=True).encode("utf-8")).hexdigest()[:16]


# Her parse_mistral_output() (notebook 3) and parse_gpt_output() (notebook 4), identical apart from the name
def parse_judge_output(raw_text):
    if not raw_text:
        return None, None, None
    try:
        parsed = json.loads(raw_text.strip())
        return parsed.get('rationale'), parsed.get('approval'), parsed.get('view_inclusion')
    except Exception:
        try:
            start = raw_text.find('{')
            end = raw_text.rfind('}') + 1
            if start != -1 and end != -1:
                parsed = json.loads(raw_text[start:end].strip())
                return parsed.get('rationale'), parsed.get('approval'), parsed.get('view_inclusion')
        except Exception:
            pass
    return None, None, None


KNOWN_JUDGE_FINGERPRINT = "aad71598e473b97e"  # the judge text and settings above, as built from her notebooks
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
JUDGE_HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://colab.research.google.com/",
    "X-Title": "LLM-as-a-Judge Evaluation",
}
JUDGE_CACHE = PATHS["judge_cache"]
MAX_ATTEMPTS = 5
STOP_STATUS = {  # errors every other request would hit too, so the cell stops instead of retrying
    401: "OpenRouter does not accept the key in OR_API_KEY.",
    402: "The OpenRouter account is out of credits. Add some at https://openrouter.ai/settings/credits.",
    404: "OpenRouter has no provider serving this judge model right now.",
}
NO_RETRY_STATUS = {400, 403}  # specific to one request (403 is a moderation flag); a retry gets the same answer


class JudgeStop(Exception):
    """An OpenRouter error that every other request would hit too."""


def judge_prompt(persona, question, response):
    """Her user prompt for one row: QUESTION is valenced_q and RESPONSE the model's response, as in her cell 11."""
    return USER_TEMPLATE.format(persona_block=PERSONAS[persona], question=question, response=response)


def as_score(value):
    """The score as a float, or None if her metric columns could not use it as a number."""
    if value is None or isinstance(value, bool):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def call_judge(model_name, user_prompt):
    """One judgment with her request. Retries the failures her code would have left blank."""
    body = json.dumps({"model": model_name,
                       "messages": [{"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                                    {"role": "user", "content": user_prompt}],
                       **JUDGE_REQUEST})
    error, attempt = "stopped because another request hit a stop error", 0
    while attempt < MAX_ATTEMPTS and not stop_requests.is_set():
        attempt += 1
        if attempt > 1:
            time.sleep(min(30, 2 ** attempt) * (0.5 + random.random()))
        try:
            reply = requests.post(OPENROUTER_URL, headers=JUDGE_HEADERS, data=body, timeout=(15, 180))
        except requests.RequestException as exc:
            error = f"{type(exc).__name__}: {exc}"
            continue
        if reply.status_code in STOP_STATUS:
            stop_requests.set()
            raise JudgeStop(f"{STOP_STATUS[reply.status_code]} (HTTP {reply.status_code}: {reply.text[:300]})")
        if reply.status_code != 200:
            error = f"HTTP {reply.status_code}: {reply.text[:300]}"
            if reply.status_code in NO_RETRY_STATUS:
                break
            continue
        try:
            payload = reply.json()
            content = payload["choices"][0]["message"]["content"]
        except (ValueError, KeyError, IndexError, TypeError):
            error = f"reply without content: {reply.text[:300]}"
            continue
        rationale, approval, view = parse_judge_output(content)
        if as_score(approval) is not None and as_score(view) is not None:
            return {"rationale": rationale, "approval": approval, "view": view, "raw": content, "attempts": attempt,
                    "provider": payload.get("provider"), "served_model": payload.get("model"),
                    "usage": payload.get("usage")}
        error = f"no scores in the reply: {content!r:.300}"
    return {"error": error, "attempts": attempt}


def load_judge_cache():
    """Finished judgments by request key; a line cut short mid-write is skipped."""
    cache = {}
    if JUDGE_CACHE.exists():
        for line in JUDGE_CACHE.read_text(encoding="utf-8").split("\n"):
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            cache[record["key"]] = record
    return cache


def request_key(judge_model, user_prompt):
    """Experiment 1's cache key for one judge request: the same text and settings give the same key."""
    return hashlib.sha256(json.dumps([judge_model, JUDGE_SYSTEM_PROMPT, user_prompt, JUDGE_REQUEST],
                                     sort_keys=True).encode("utf-8")).hexdigest()


def run_gradings(tasks):
    """Send every grading not yet in the cache, as Experiment 1 did. Returns {task key: error} for failures."""
    cache = load_judge_cache()
    todo = [task for task in tasks if task["key"] not in cache]
    print(f"{len(tasks) - len(todo)} of {len(tasks)} gradings already cached; {len(todo)} to request.")
    failures = {}
    if not todo:
        return failures
    stop_requests.clear()
    started = time.time()
    with ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool, open_for_append(JUDGE_CACHE) as handle:
        pending = {pool.submit(call_judge, task["judge_model"], task["prompt"]): task for task in todo}
        try:
            for finished in tqdm(as_completed(pending), total=len(pending), desc="gradings"):
                task = pending[finished]
                result = finished.result()
                if "error" in result:
                    failures[task["key"]] = result["error"]
                    continue
                record = {"key": task["key"], "judge": task["judge"], "judge_model": task["judge_model"],
                          "model": task["model"], "id": task["id"], "type": task["type"],
                          "graded_utc": utc_now(), **result}
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                handle.flush()
        except BaseException as exc:  # a stop error, or you interrupting the cell: cancel what has not started
            stop_requests.set()
            for future in pending:
                future.cancel()
            if isinstance(exc, JudgeStop):
                raise RuntimeError(f"{exc} Finished gradings are cached; fix this, then run this cell again.") from None
            raise
    print(f"Requested {len(todo)} gradings in {(time.time() - started) / 60:.1f} min.")
    return failures


stop_requests = threading.Event()
if JUDGE_FINGERPRINT == KNOWN_JUDGE_FINGERPRINT:
    print(f"Judge fingerprint {JUDGE_FINGERPRINT}: Experiment 1's prompts and settings, unchanged")
else:
    raise RuntimeError(f"Judge fingerprint {JUDGE_FINGERPRINT} differs from Experiment 1's {KNOWN_JUDGE_FINGERPRINT}: "
                       "the judge text or settings were edited, so these gradings would not be comparable.")

Judge fingerprint aad71598e473b97e: Experiment 1's prompts and settings, unchanged


In [ ]:
judge_tasks = []  # one per (answer to a rewording, judge)
for row in answers[answers["type"] != "original"].itertuples(index=False):
    for prefix, (persona, judge_model) in JUDGES.items():
        prompt = judge_prompt(persona, str(row.valenced_q), str(row.response))  # valenced_q: the original question
        judge_tasks.append({"key": f"{request_key(judge_model, prompt)}|{row.model}|{row.id}|{row.type}",
                            "judge": prefix, "judge_model": judge_model, "model": row.model, "id": row.id,
                            "type": row.type, "prompt": prompt})
random.Random(SEED).shuffle(judge_tasks)  # the same send order in every session

grading_failures = run_gradings(judge_tasks)
if grading_failures:
    print(f"\nWARNING: {len(grading_failures)} gradings failed. Run this cell again to retry only those. Examples:")
    for key, error in list(grading_failures.items())[:5]:
        print(f"  {key.split('|', 1)[1]}: {error}")

0 of 11200 gradings already cached; 11200 to request.


gradings:   0%|          | 0/11200 [00:00<?, ?it/s]

Requested 11200 gradings in 51.5 min.

  Gemma-3-12b-it|abortion_C_1|paraphrase: no scores in the reply: '{"rationale": "The response provides a clear and respectful explanation of the legal distinctions between abortion and the " \t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t


In [ ]:
MODEL_ORDER = {spec["name"]: number for number, spec in enumerate(CANDIDATES)}
QUESTION_ORDER = {prompt_id: number for number, prompt_id in enumerate(wordings(versions, "original")["id"])}
TYPE_ORDER = {kind: number for number, kind in enumerate(WORDING_TYPES)}
JUDGE_ORDER = {prefix: number for number, prefix in enumerate(JUDGES)}


def in_study_order(frame):
    """Rows by model (notebook order), question (Experiment 1's order), wording (original first), then judge."""
    ranks = {"_model": frame["model"].map(MODEL_ORDER), "_id": frame["id"].map(QUESTION_ORDER)}
    for column, order in (("type", TYPE_ORDER), ("judge", JUDGE_ORDER)):
        if column in frame:
            ranks[f"_{column}"] = frame[column].map(order)
    return (frame.assign(**ranks).sort_values(list(ranks), kind="stable")
            .drop(columns=list(ranks)).reset_index(drop=True))


def as_number(value):
    score = as_score(value)
    return math.nan if score is None else score


judge_cache = load_judge_cache()
grading_rows = []
for task in judge_tasks:
    record = judge_cache.get(task["key"]) or {}
    usage = record.get("usage") or {}
    grading_rows.append({"model": task["model"], "id": task["id"], "type": task["type"], "judge": task["judge"],
                         "approval": as_number(record.get("approval")), "view": as_number(record.get("view")),
                         "rationale": record.get("rationale"), "provider": record.get("provider"),
                         "served_model": record.get("served_model"), "attempts": record.get("attempts"),
                         "graded_utc": record.get("graded_utc"), "prompt_tokens": usage.get("prompt_tokens"),
                         "completion_tokens": usage.get("completion_tokens")})
gradings = in_study_order(pd.DataFrame(grading_rows))
gradings.to_csv(PATHS["gradings"], index=False)

grading_summary = (gradings.groupby("judge", sort=False)
                   .agg(requested=("approval", "size"), scored=("approval", "count"),
                        retried=("attempts", lambda s: int((s > 1).sum())),
                        outside_0_1=("approval", lambda s: int(((s < 0) | (s > 1)).sum())),
                        prompt_tokens=("prompt_tokens", "sum"), completion_tokens=("completion_tokens", "sum"))
                   .reset_index())
PROVIDERS = {judge: dict(Counter(g["provider"].fillna("unknown"))) for judge, g in gradings.groupby("judge")}
print(f"{int(gradings['approval'].notna().sum())} of {len(gradings)} gradings scored.")
print("Served by:", PROVIDERS)
if grading_summary["outside_0_1"].sum():
    print("WARNING: some approval scores fall outside 0-1. They stay as the judge gave them, as in Experiment 1.")
grading_summary

11199 of 11200 gradings scored.
Served by: {'c_gpt': {'OpenAI': 2797, 'Azure': 2803}, 'l_mistral': {'Venice': 2380, 'DeepInfra': 3209, 'Parasail': 10, 'unknown': 1}}


,judge,requested,scored,retried,outside_0_1,prompt_tokens,completion_tokens
0,l_mistral,5600,5599,1,0,5180068.0,465862.0
1,c_gpt,5600,5600,0,0,4964447.0,469190.0


## 6. The scores: every model, question, and wording

In [ ]:
JUDGE_SCORES = ["l_mistral_approval", "c_gpt_approval", "l_mistral_view", "c_gpt_view"]
SCORE_COLUMNS = (["model", "id", "cluster", "topic", "valence", "type", "question", "response", "refusal_like"]
                 + JUDGE_SCORES + ["delta", "graded_in"])


def original_scores():
    """Experiment 1's 1,400 answers to the originals, with its one grading of each."""
    frame = pd.concat([table.assign(model=name, type="original")[["model", "id", "type", "response"]
                                                                  + EXP1_SCORE_COLUMNS]
                       for name, table in exp1.items()], ignore_index=True)
    return frame.assign(**{column: pd.to_numeric(frame[column], errors="coerce") for column in EXP1_SCORE_COLUMNS},
                        graded_in="Experiment 1")


def rewording_scores():
    """This run's 5,600 answers to rewordings, with both judges' scores side by side."""
    wide = gradings.pivot(index=["model", "id", "type"], columns="judge", values=["approval", "view"])
    wide.columns = [f"{judge}_{value}" for value, judge in wide.columns]
    frame = answers.loc[answers["type"] != "original", ["model", "id", "type", "response"]]
    return frame.join(wide, on=["model", "id", "type"]).assign(graded_in="Stage 1")


def wording_summary(frame):
    return {"rows": len(frame), "with_delta": int(frame["delta"].notna().sum()),
            "refusal_like": int(frame["refusal_like"].sum()), "mean_delta": float(frame["delta"].mean())}


scores = pd.concat([original_scores(), rewording_scores()], ignore_index=True)
scores = scores.merge(versions[["id", "type", "cluster", "topic", "valence", "question"]], on=["id", "type"],
                      how="left", validate="many_to_one")
scores = in_study_order(scores.assign(refusal_like=refusal_like(scores["response"]),
                                      delta=scores["l_mistral_approval"] - scores["c_gpt_approval"])[SCORE_COLUMNS])
deltas = (scores.pivot(index=["model", "id"], columns="type", values="delta")[list(WORDING_TYPES)]
          .add_prefix("delta_").reset_index()
          .merge(wordings(versions, "original")[["id", "topic", "valence"]], on="id", how="left"))
deltas = in_study_order(deltas[["model", "id", "topic", "valence"] + [f"delta_{kind}" for kind in WORDING_TYPES]])

scores.to_csv(PATHS["scores"], index=False)
save_workbook(PATHS["scores_xlsx"], {"scores": scores, "deltas": deltas}, check_sheet="scores")
missing_delta = int(scores["delta"].isna().sum())
print(f"{len(scores)} rows saved to {PATHS['scores'].name} and {PATHS['scores_xlsx'].name}; "
      f"{len(scores) - missing_delta} have both judges' approval scores")
if missing_delta:
    print(f"WARNING: {missing_delta} rows lack a score because a grading failed. Run the grading cell in section 5 "
          "again, then the cells after it.")
print("\nFirst look, mean δ by model and wording (the analysis notebook tests whether rewording moves it):")
(scores.pivot_table(index="model", columns="type", values="delta", aggfunc="mean")
 .reindex(index=[spec["name"] for spec in CANDIDATES], columns=list(WORDING_TYPES)).round(3))

7000 rows saved to scores.csv and scores.xlsx; 6999 have both judges' approval scores

First look, mean δ by model and wording (the analysis notebook tests whether rewording moves it):


type,original,typo,synonym,restructure,paraphrase
model,,,,,
Llama-3.1-8B-Instruct,0.242,0.242,0.233,0.232,0.231
Gemma-3-12b-it,0.226,0.223,0.215,0.227,0.241
Qwen3-VL-8B-Instruct,0.366,0.383,0.348,0.360,0.357
GLM-4-9b-chat,0.229,0.220,0.230,0.216,0.235
Llama-SEA-LION-v3-8B-IT,0.256,0.256,0.246,0.251,0.252
Qwen-SEA-LION-v4-8B-VL,0.349,0.345,0.341,0.354,0.341
Llama3.1-typhoon2-8b-instruct,0.223,0.241,0.257,0.238,0.247


## 7. Summary

In [ ]:
def json_safe(value):
    """Plain Python values for json.dumps, with NaN written as null."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


summary = {
    "answers": {"total": len(answers), "versions_fingerprint": KNOWN_VERSIONS_FINGERPRINT,
                "originals_identical_to_exp1": {run["model"]["name"]: run["reproduced"] for run in runs},
                "runs": runs},
    "gradings": {"requested": len(gradings), "scored": int(gradings["approval"].notna().sum()),
                 "per_judge": grading_summary.to_dict("records"), "providers": PROVIDERS,
                 "judge_fingerprint": JUDGE_FINGERPRINT},
    "scores": {"rows": len(scores), "with_delta": int(scores["delta"].notna().sum()),
               "by_wording": {kind: wording_summary(scores[scores["type"] == kind]) for kind in WORDING_TYPES}},
    "settings": {"candidates": CANDIDATES, "system_prompt": SYSTEM_PROMPT, "sampling": SAMPLING,
                 "batch_size": BATCH_SIZE, "seed": SEED,
                 "judges": {"liberal": LIBERAL_JUDGE, "conservative": CONSERVATIVE_JUDGE},
                 "judge_request": JUDGE_REQUEST, "judge_question": "valenced_q (the original question)",
                 "originals_graded_in": "Experiment 1", "require_exact_reproduction": REQUIRE_EXACT_REPRODUCTION},
    "saved_utc": utc_now(),
}
PATHS["summary"].write_text(json.dumps(json_safe(summary), indent=2, default=str))
print(f"Answers: {len(answers)} | gradings scored: {summary['gradings']['scored']} of {len(gradings)} | "
      f"scores with δ: {summary['scores']['with_delta']} of {len(scores)}")
print(f"\nFiles in {BASE_DIR}:", ", ".join(sorted(p.name for p in BASE_DIR.iterdir())))

Answers: 7000 | gradings scored: 11199 of 11200 | scores with δ: 6999 of 7000

Files in /content/drive/MyDrive/CS612_stage1: answers, answers.jsonl, answers.xlsx, exp1, generation_checkpoints, gradings.csv, judge_cache.jsonl, scores.csv, scores.xlsx, stage1_summary.json
